# Color Shift Analyzer

Measure how a cross-processed image shifts color across **shadows, midtones, and highlights** — the tone-dependent behaviour that a single color grade can't capture.


In [ ]:
# Make the tools/ package importable from the notebooks/ folder.
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "tools")))
import numpy as np
import matplotlib.pyplot as plt
import xpro_core as xc
print("presets:", xc.list_presets())


In [ ]:
def sample_image(h=256, w=256):
    """A synthetic test image: gradients + color patches, no files needed."""
    y, x = np.mgrid[0:h, 0:w].astype(np.float32)
    r = x / (w - 1)
    g = y / (h - 1)
    b = (1 - r + g) / 2
    img = np.stack([r, g, b], -1)
    # drop in a few saturated patches and a gray ramp
    img[20:60, 20:60] = [0.9, 0.1, 0.1]
    img[20:60, 80:120] = [0.1, 0.7, 0.2]
    img[20:60, 140:180] = [0.2, 0.3, 0.9]
    img[h-50:h-20, :] = np.linspace(0, 1, w)[None, :, None]
    return np.clip(img, 0, 1)

img = sample_image()
plt.figure(figsize=(4,4)); plt.imshow(img); plt.title("synthetic input"); plt.axis("off");


## Make a cross-processed version to analyze
(In practice you'd load a real scan with `xc.load_image(...)`.)

In [ ]:
xpro = xc.apply_profile(img, xc.PRESETS["push_xpro"], seed=4)
plt.imshow(xpro); plt.axis("off"); plt.title("cross-processed sample");


## Per-tone color cast

In [ ]:
def tone_means(rgb):
    flat = rgb.reshape(-1, 3)
    l = xc.luminance(flat)
    regions = {"shadows": l < 0.33, "midtones": (l>=0.33)&(l<=0.66), "highlights": l>0.66}
    return {k: flat[m].mean(0) for k, m in regions.items() if m.any()}

before, after = tone_means(img), tone_means(xpro)
for region in after:
    shift = after[region] - before.get(region, after[region])
    print(f"{region:10s} mean={after[region].round(3)}  shift={shift.round(3)}")


## Visualize the shift as RGB curves vs. tone

In [ ]:
bins = np.linspace(0, 1, 32)
l = xc.luminance(img).ravel()
idx = np.digitize(l, bins) - 1
flat_in, flat_out = img.reshape(-1,3), xpro.reshape(-1,3)
curve = np.zeros((len(bins), 3))
for b in range(len(bins)):
    m = idx == b
    if m.any(): curve[b] = (flat_out[m] - flat_in[m]).mean(0)
for ci, color in enumerate(["r","g","b"]):
    plt.plot(bins, curve[:, ci], color, label=color.upper())
plt.axhline(0, color="k", lw=0.5); plt.xlabel("input luminance"); plt.ylabel("color shift")
plt.title("Tone-dependent color shift"); plt.legend();


## Export an analysis profile
Reuse the extractor to turn the measured shift into a portable profile.

In [ ]:
from xpro_profile_extractor import extract_paired
prof = extract_paired(img, xpro, name="analyzed")
prof.to_json("analyzed.json")
print(prof)
